In [ ]:
import numpy as np
import SimpleITK as sitk
from typing import Tuple
import skimage as sk
from PIL import Image
import napari
import matplotlib.pyplot as plt
import scipy as sc
from scipy.ndimage import zoom
from glob import glob
#from Get_Orientation import get_cardinal_points, detect_cardinal_point, compute_transform
import pandas as pd
import os
import gc
import json
from tqdm import tqdm

In [ ]:
def get_tissue_bbox(img,name,tissue_id,data_path):
    mask = img < 255
    mask = sc.ndimage.binary_fill_holes(mask)
    labels = sk.measure.label(mask)
    props = sk.measure.regionprops(labels)
    largest_obj = max(props, key=lambda p: p.area)
    largest_obj_label = largest_obj.label
    minc = min(props, key=lambda p: p.bbox[0])
    minr = min(props, key=lambda p: p.bbox[1])
    maxc = max(props, key=lambda p: p.bbox[2])
    maxr = max(props, key=lambda p: p.bbox[3])
    img_bbox = img[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
    labels_bbox = labels[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
    cropped_mask = (labels_bbox==largest_obj_label)*labels_bbox
    save_path_img = os.path.join(data_path,tissue_id,'cropped_image')
    save_path_mask = os.path.join(data_path,tissue_id,'cropped_mask')
    os.makedirs(save_path_img,exist_ok=True)
    os.makedirs(save_path_mask,exist_ok=True)
    sk.io.imsave(os.path.join(save_path_img,'cropped_'+name+'.png'),img_bbox,check_contrast=False)
    sk.io.imsave(os.path.join(save_path_mask,'cropped_mask_'+name+'.png'),cropped_mask,check_contrast=False)
    return img_bbox, cropped_mask



In [ ]:
def add_padding(img_bbox,cropped_mask,name,tissue_id,data_path):
    """ 
    reshape the cropped images to a square based on the largest dim using padding
    saves new arrays with added padding of a constant value: 255 for grey scale image, 0 for masks
    returns save path for later use
    """
    max_dim = max(img_bbox.shape)
    padding_y = (max_dim - img_bbox.shape[0]) // 2
    padding_x = (max_dim - img_bbox.shape[1]) // 2
    pad_img_bbox = np.pad(img_bbox,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=255)
    pad_mask_bbox = np.pad(cropped_mask,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=0)
    save_path_img = os.path.join(data_path,tissue_id,'padded_cropped_image')
    save_path_mask = os.path.join(data_path,tissue_id,'padded_cropped_mask')
    os.makedirs(save_path_img,exist_ok=True)
    os.makedirs(save_path_mask,exist_ok=True)
    sk.io.imsave(os.path.join(save_path_img,'padded_cropped_'+name+'.png'),pad_img_bbox,check_contrast=False)
    sk.io.imsave(os.path.join(save_path_mask,'padded_cropped_mask_'+name+'.png'),pad_mask_bbox,check_contrast=False)
    return pad_img_bbox,pad_mask_bbox



In [ ]:
def get_map_region(data_path,map_base,map_id,grey_value_df):
    path = os.path.join(data_path,map_base+'.png')
    map_arr = np.array(Image.open(path).convert('L'))
    grey_value = grey_value_df.loc[grey_value_df['Mapping_ID'] == map_id, 'Map_Grey_value'].values[0] 
    map_region = (map_arr == int(grey_value)).astype(np.float32)
    map_region = sc.ndimage.binary_fill_holes(map_region)
    return map_region

In [ ]:
def detect_cardinal_point(arr, grey_value, name):
    """
    Detect the centroid of a cardinal point marker by its grey value.
    
    Returns (x, y) in pixel coordinates, or None if not found.
    """
    
    # Create binary mask for this grey value
    mask = (arr * (arr == int(grey_value))).astype(int)
        
    if not mask.any():
        print(f"  WARNING: No pixels found for grey value {grey_value} in {name}")
        return None
    
    # Label connected components and get centroid
    labeled_img = sk.measure.label(mask)
    props = sk.measure.regionprops(labeled_img)
    cy,cx = props[0].centroid
    return np.array([int(cy), int(cx)])  # return as (y,x)


In [ ]:
def get_cardinal_points(img,name,grey_value_df):
    """
    Extract both cardinal points from an image.
    Returns dict with 'north' and 'east' as (y,x) arrays.
    """
    points = {}
    for direction in ["north", "east"]:
        grey = grey_value_df.loc[grey_value_df['Mapping_ID'] == direction, 'Tissue_Grey_value'].values[0]
        print(f'Grey value for {direction} is {grey}')
        pt = detect_cardinal_point(img, grey, name)
        if pt is None:
            raise ValueError(f"Could not find '{direction}' point in {name}")
        points[direction] = pt
        print(f"  {direction}: pixel (y,x) ({pt[0]:.1f}, {pt[1]:.1f})")
    return points


In [ ]:
def orient_tissue(mask_points,moving_image):
    """
    Determine the flip and/or rotation needed to orient the moving image
    to match the map, using the N and E cardinal point locations.

    Detection logic (image coordinates, y increases downward):
        North marker should be visually "above" East  → north_y < east_y
        East  marker should be visually "right" of North → east_x > north_x

    Flip cases:
        north_is_up  and east_is_right  → none
        north_is_up  and east_is_left → horizontal flip
        north_is_right and east_is_up  → -90 rotation
        north_is_right and east_is_down → 90 rotation
        north_is_left and east_is_up → Vertical flip + 90 rotation
        north_is_left and east_is_down → Vertical flip + -90 rotation
        north_is_down and east_is_left → vertical + horizontal flip
        north_is_down and east_is_right → vertical flip

    Parameters
    ----------
    mask_points  : dict with 'north' and 'east' as (x, y) pixel arrays — moving image
    moving_image: image array of moving image
    Returns
    -------
    flip matrix
    """
    # --- Detect required flip from cardinal point geometry ---
    # Compare pixel coords directly for orientation — spacing does not
    # affect which direction is "up" or "right"
    #get moving image relative cardinal coordinates:
    north = (0, moving_image.shape[1]//2)
    south = (moving_image.shape[0]-1, moving_image.shape[1]//2)
    east = (moving_image.shape[0]//2, moving_image.shape[1]-1)
    west = (moving_image.shape[0]//2, 0)
    cardinals = {"north": north, "south": south, "east": east, "west": west}

    #which cardinal direction are the two reference points closest to?
    closest_north = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['north'][0] - cardinals[d][0], 
                    mask_points['north'][1] - cardinals[d][1]))
    closest_east = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['east'][0] - cardinals[d][0], 
                    mask_points['east'][1] - cardinals[d][1]))

    if closest_north == 'north' and closest_east == 'east':
        flip = None
        rotation = None
    elif closest_north == 'north' and closest_east == 'west':
        flip = 'horizontal'
        rotation = None
    elif closest_north == 'west' and closest_east == 'north':
        flip = None
        rotation = 'clockwise'
    elif closest_north == 'east' and closest_east == 'north':
        flip = 'vertical'
        rotation = 'counter-clockwise'
    elif closest_north == 'west' and closest_east == 'south':
        flip = 'vertical'
        rotation = 'clockwise'
    elif closest_north == 'east' and closest_east == 'south':
        flip = None
        rotation = 'counter-clockwise'
    elif closest_north == 'south' and closest_east == 'west':
        flip = 'both'
        rotation = None
    elif closest_north == 'south' and closest_east == 'east':
        flip = 'vertical'
        rotation = None
    else:
        print('Orientation does not match conditions, check image')
    print(f'Detected Flip: {flip}')
    print(f'Detected Rotation: {rotation}')
    return flip, rotation

In [ ]:
def get_mask_centroid(sitk_image):
    """Extract physical centroid of the mask region."""
    binary = sitk.BinaryThreshold(
        sitk_image,
        lowerThreshold= 0.5,
        upperThreshold= 255.0,
        insideValue=1,
        outsideValue=0
    )
    label_stats = sitk.LabelShapeStatisticsImageFilter()
    label_stats.Execute(binary)
    if 1 not in label_stats.GetLabels():
        raise ValueError("No mask found — check your threshold.")
    return label_stats.GetCentroid(1)  # returns physical coords


def get_centroid_alignment_transform(sitk_moving, sitk_fixed):
    """
    Build a translation transform that maps the moving mask centroid
    to the fixed mask centroid, replacing CenteredTransformInitializer.
    """
    fixed_centroid  = get_mask_centroid(sitk_fixed)
    moving_centroid = get_mask_centroid(sitk_moving)

    translation = sitk.TranslationTransform(sitk_moving.GetDimension())
    # SimpleITK transforms map fixed→moving, so the offset is inverted
    offset = [m - f for f, m in zip(fixed_centroid, moving_centroid)]
    translation.SetOffset(offset)
    return translation

In [ ]:
def apply_flip_rotation(sitk_moving, sitk_fixed, flip=None, rotation=None):
    FLIP_MATRICES = {
        "none":       [ 1,  0,  0,  1],
        "horizontal": [-1,  0,  0,  1],
        "vertical":   [ 1,  0,  0, -1],
        "both":       [-1,  0,  0, -1],
    }
    ROTATION_DICTIONARY = {
        'clockwise': -90,
        'counter-clockwise': 90
    }

    composite_transform = sitk.CompositeTransform(2)

    # --- Flip ---
    if flip is not None:
        print(f"  Applying flip: {flip}")
        moving_center = sitk_moving.TransformContinuousIndexToPhysicalPoint(
            [(sz - 1) / 2.0 for sz in sitk_moving.GetSize()]
        )
        flip_transform = sitk.AffineTransform(sitk_moving.GetDimension())
        flip_transform.SetCenter(moving_center)
        flip_transform.SetMatrix(FLIP_MATRICES[flip])
        composite_transform.AddTransform(flip_transform)

    # --- Rotation ---
    if rotation is not None:
        print(f"  Applying rotation: {rotation}")
        moving_center = sitk_moving.TransformContinuousIndexToPhysicalPoint(
            [(sz - 1) / 2.0 for sz in sitk_moving.GetSize()]
        )
        angle = ROTATION_DICTIONARY[rotation]
        rads  = np.deg2rad(angle)
        rotation_transform = sitk.Euler2DTransform()
        rotation_transform.SetCenter(moving_center)
        rotation_transform.SetAngle(rads)
        composite_transform.AddTransform(rotation_transform)

    if flip is None and rotation is None:
        print("  No flip or rotation applied.")

    # --- Centroid alignment: moving mask → fixed mask (KEY FIX) ---
    centroid_transform = get_centroid_alignment_transform(sitk_moving, sitk_fixed)
    composite_transform.AddTransform(centroid_transform)

    return composite_transform

In [ ]:
def to_distance_map(sitk_img):
    """Convert binary mask to signed Maurer distance map for smoother optimization."""
    binary = sitk.BinaryThreshold(sitk_img, lowerThreshold=0.5, upperThreshold=255.0,
                                  insideValue=1, outsideValue=0)
    return sitk.SignedMaurerDistanceMap(
        sitk.Cast(binary, sitk.sitkUInt8),
        insideIsPositive=True,
        squaredDistance=False,
        useImageSpacing=True
    )

In [ ]:
def refine_registration(sitk_moving,sitk_fixed,composite_transform,data_path,tissue_id,name):
    save_path_tfm = os.path.join(data_path,tissue_id,'transformation_files','tfm')
    save_file_tfm = os.path.join(save_path_tfm,f"TF_tfm_{name}_{tissue_id}.tfm")
    save_path_mx = os.path.join(data_path,tissue_id,'transformation_files','csv')
    save_file_mx = os.path.join(save_path_mx,f"TF_matrix_{name}_{tissue_id}.csv")
    os.makedirs(save_path_tfm,exist_ok=True)
    os.makedirs(save_path_mx,exist_ok=True)

    coarse_resampled = sitk.Resample(
        sitk_moving, sitk_fixed, composite_transform,
        sitk.sitkNearestNeighbor, 0.0, sitk_moving.GetPixelID()
    )

    moving_dist = to_distance_map(coarse_resampled)
    fixed_dist = to_distance_map(sitk_fixed)

    fixed_center = sitk_fixed.TransformContinuousIndexToPhysicalPoint(
        [(sz - 1) / 2.0 for sz in sitk_fixed.GetSize()]
    )
    fine_transform = sitk.Similarity2DTransform()  # 4 DOF: angle, scale, tx, ty
    fine_transform.SetCenter(fixed_center)

    registration_method = sitk.ImageRegistrationMethod()
    fixed_mask_sitk = sitk.BinaryThreshold(sitk_fixed, lowerThreshold=0.5,
                                        upperThreshold=255.0, insideValue=1, outsideValue=0)
    registration_method.SetMetricFixedMask(fixed_mask_sitk)
    registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
    registration_method.SetMetricSamplingPercentage(0.2)
    registration_method.SetMetricAsJointHistogramMutualInformation()
    registration_method.SetInitialTransform(fine_transform, inPlace=False)
    registration_method.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=500,
        convergenceMinimumValue=1e-6,
        convergenceWindowSize=20
    )
    registration_method.SetOptimizerScalesFromPhysicalShift()
    registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[2, 1])
    registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[1, 0])
    registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    registration_method.SetInterpolator(sitk.sitkLinear)
    print(f"  Registering...")
    fine_result = registration_method.Execute(fixed_dist, moving_dist)
    metric = registration_method.GetMetricValue()
    print(f"metric: {metric:.6f}")
    full_composite = sitk.CompositeTransform(2)
    full_composite.AddTransform(composite_transform)
    full_composite.AddTransform(fine_result)
    #save transform as tfm and csv
    full_composite.FlattenTransform()
    sitk.WriteTransform(full_composite,save_file_tfm)

    p0 = full_composite.TransformPoint((0.0, 0.0))
    px = full_composite.TransformPoint((1.0, 0.0))
    py = full_composite.TransformPoint((0.0, 1.0))
    M = np.array([
        [px[0] - p0[0],  py[0] - p0[0],  p0[0]],
        [px[1] - p0[1],  py[1] - p0[1],  p0[1]],
        [0.0,            0.0,             1.0  ]
    ])
    pd.DataFrame(M).to_csv(save_file_mx, index=False, header=False)

    return full_composite

In [ ]:
def calculate_spacing(img_mask,map_region_mask):
    #get area of tissue mask bounding box
    moving_labels = sk.measure.label(img_mask)
    props = sk.measure.regionprops(moving_labels)
    minc,minr,maxc,maxr = props[0].bbox
    moving_height_px = maxr - minr
    moving_width_px  = maxc - minc
    #moving_area_px = moving_height_px * moving_width_px
    #get area of map region bounding box
    map_region_label = sk.measure.label(map_region_mask)
    map_region_props = sk.measure.regionprops(map_region_label)
    minc,minr,maxc,maxr = map_region_props[0].bbox
    map_height_px = maxr - minr
    map_width_px  = maxc - minc
    #map_area_px = map_height_px * map_width_px
    height_spacing = moving_height_px / map_height_px
    width_spacing = moving_width_px / map_width_px
    map_spacing = (width_spacing,height_spacing)
    return map_spacing

In [ ]:
def load_sitk_moving_img(moving_img_arr,moving_spacing):
    """ Convert array to 32bit float and then to sitk image for registration """
    bitdepth = np.array(moving_img_arr).astype(np.float32)
    sitk_img = sitk.GetImageFromArray(bitdepth)
    sitk_img.SetSpacing(moving_spacing)
    return sitk_img

def load_sitk_fixed_img(fixed_img_arr,fixed_spacing):
    """ Convert array to 32bit float and then to sitk image for registration """
    bitdepth = np.array(fixed_img_arr).astype(np.float32)
    sitk_img = sitk.GetImageFromArray(bitdepth)
    sitk_img.SetSpacing(fixed_spacing)
    return sitk_img
    

Mammary gland sizes:
Small gland:
width: 4.1cm
height: 5.3 cm
Large gland:
Width: 4.3 cm
Height: 6.1 cm

In [ ]:
#get mapID grey values
maps = sorted(glob('Maps\PNGsForReg\GrayscaleMaps\*.png'))
names = list(map(os.path.basename,maps))
map_arrays = [np.array(Image.open(map).convert('L')) for map in maps]
test_map = map_arrays[0]



In [ ]:
avg_y = round(float(np.average([5.3,6.1])),2)
avg_x = round(float(np.average([4.1,4.3])),2)

In [ ]:
#get spacing for maps:
map = np.array(Image.open(r'Pipeline_Test\Maps\SD211_Left.png').convert('L'))
map_mask = map < 255
map_labels = sk.measure.label(map_mask)
props = sk.measure.regionprops(map_labels)
map_region = max(props, key=lambda p: p.area)
minc,minr,maxc,maxr = map_region.bbox
map_height_px = maxr - minr
map_width_px  = maxc - minc
y_spacing_um = round((avg_y / map_height_px) * 10000, 2)
x_spacing_um = round((avg_x / map_width_px)  * 10000, 2)

In [ ]:
target_spacing = 16.1
zoom_y = y_spacing_um/target_spacing
zoom_x = x_spacing_um/target_spacing

In [ ]:
isotropic_maps = [zoom(map, (zoom_y,zoom_x),order=0) for map in map_arrays]

In [ ]:
padded_maps = [np.pad(im,(100,100),mode='constant',constant_values=255) for im in isotropic_maps]
#viewer = napari.view_image(padded_map)

In [ ]:
save_dir = r'Maps/PNGsForReg/isotropic_maps'
for m, name in zip(padded_maps,names):
    save = os.path.join(save_dir,name)
    Image.fromarray(m).save(save)

In [ ]:
Image.fromarray(padded_map).save(r'Pipeline_Test\Maps\Isotropic_Maps\SD211-Left.png')

In [ ]:
pixel_size = {
    'map spacing' : {'yspacing': 16.1,
    'xspacing': 16.1},
    'image spacing' : {'yspacing': 16.1,
    'xspacing': 16.1}
}

In [ ]:
pixel_size

In [ ]:
import json
file = 'pixel_spacing.json'
with open(file, 'w') as json_file:
    json.dump(pixel_size, json_file, indent=4)

In [ ]:
for i, name in enumerate(names):
    print(f'{i}. {name}')

In [ ]:
largest_array = max(map_arrays, key=lambda p: p.shape[0]*p.shape[1])
smallest_array = min(map_arrays, key=lambda p: p.shape[0]*p.shape[1])
print(largest_array.shape, smallest_array.shape)

In [ ]:
matched_size = [sk.transform.resize(map,(1100,1000),mode='constant',cval=255,preserve_range=True) for map in map_arrays]

In [ ]:
stacked_maps = np.stack(matched_size)
viewer = napari.view_image(stacked_maps)

In [ ]:
#set up save locations and load dataframes:
data_path = 'Pipeline_Test'
maps_path = r'Maps/PNGsForReg/isotropic_maps'
df_key = pd.read_csv('2026.3.4.AnnotationlevelKey.csv',dtype=str,usecols=['Image','Genotype','Tissue.ID','MappingID','MapBase','AnimalID'])
grey_value_df = pd.read_csv('Maps\PNGsForReg\HexColorMappingKey.2026.2.28.csv',dtype=str,usecols=['Mapping_ID','Map_Grey_value','Tissue_Grey_value'])
with open('pixel_spacing.json', 'r') as json_file:
    pixel_spacing = json.load(json_file)


In [ ]:
df_key['MapBase'] = df_key['MapBase'].str.replace(' ', '').str.replace('-', '_')

In [ ]:
pixel_spacing

In [ ]:
test_df = df_key[df_key['AnimalID'] == 'SD178']

In [ ]:
test_df.tail(n=20)

In [ ]:
flip_series = []
rotation_series = []
save_path_cropped_masks = []
padded_imgs = []
padded_masks = []

image_names = test_left_df['Image'].to_list()
tissue_ids = test_left_df['Tissue.ID'].to_list()
map_ids = test_left_df['MappingID'].to_list()
map_base = test_left_df['MapBase'].to_list()

# test_name = str(image_names[0])
# test_tissue_id = tissue_ids[0]
# test_map_id = map_ids[0]
# map = map_base[0]

In [ ]:
img = np.array(Image.open(r'Pipeline_Test\Maps\Isotropic_Maps\SD211-Left.png').convert('L'))
viewer = napari.view_image(img)

In [ ]:
fixed_spacing = (1.0,1.0)
moving_spacing = (1.1,1.1)
transforms = []
sitk_moving_imgs = []
sitk_fixed_imgs = []
for i, row in test_df.iterrows():
    img_path = os.path.join(data_path,row['Tissue.ID'],row['Image']+'.svs.png')
    img = np.array(Image.open(img_path).convert('L'))
    img_bbox, cropped_mask = get_tissue_bbox(img,row['Image'],row['Tissue.ID'],data_path)
    pad_img_bbox, pad_mask_bbox = add_padding(img_bbox,cropped_mask,row['Image'],row['Tissue.ID'],data_path)
    points = get_cardinal_points(img_bbox,row['Image'],grey_value_df)
    #zoom_factor = 4.06/16.1
    #zoom_mask = zoom(pad_mask_bbox,(zoom_factor,zoom_factor),order=0)
    flip, rotation = orient_tissue(points,img_bbox)
    map_region = get_map_region(maps_path,row['MapBase'],row['MappingID'],grey_value_df)
    #fixed_spacing = calculate_spacing(pad_mask_bbox,map_region)
    #print(f'Fixed Spacing: {fixed_spacing}')
    sitk_moving = load_sitk_moving_img(pad_mask_bbox,moving_spacing)
    sitk_fixed = load_sitk_fixed_img(map_region,fixed_spacing)
    composite_transform = apply_flip_rotation(sitk_moving,sitk_fixed,flip,rotation)
    transform = refine_registration(sitk_moving,sitk_fixed,composite_transform,data_path,row['Tissue.ID'],row['Image'])
    transforms.append(transform)
    sitk_moving_imgs.append(sitk_moving)
    sitk_fixed_imgs.append(sitk_fixed)


In [ ]:
#set up loop as function:
def process_sample(row,data_path,grey_value_df,pixel_spacing):
    try:
        spacing = (16.1,16.1)
        img_path = os.path.join(data_path, row['Tissue.ID'], row['Image'] + '.svs.png')
        img = np.array(Image.open(img_path).convert('L'))
        img_bbox, cropped_mask = get_tissue_bbox(img, row['Image'], row['Tissue.ID'], data_path)
        pad_img_bbox, pad_mask_bbox = add_padding(img_bbox, cropped_mask, row['Image'], row['Tissue.ID'], data_path)
        points = get_cardinal_points(img_bbox, row['Image'], grey_value_df)
        flip, rotation = orient_tissue(points, img_bbox)
        map_region = get_map_region(data_path, row['MapBase'], row['MappingID'], grey_value_df)
        sitk_moving = load_sitk_moving_img(pad_mask_bbox, moving_spacing)
        sitk_fixed  = load_sitk_fixed_img(map_region, fixed_spacing)
        composite_transform = apply_flip_rotation(sitk_moving, sitk_fixed, flip, rotation)
        refine_registration(sitk_moving, sitk_fixed, composite_transform,
                                        data_path, row['Tissue.ID'], row['Image'])
        return row['Image'], 'success', None
    except Exception as e:
        return row['Image'], 'failed', str(e)


In [ ]:
sitk_moving_arr = sitk.GetArrayFromImage(sitk_moving)
sitk_fixed_arr = sitk.GetArrayFromImage(sitk_fixed)

In [ ]:
viewer = napari.view_image(sitk_fixed_arr)

In [ ]:
resampled1 = sitk.Resample(
        sitk_moving_imgs[0], sitk_fixed_imgs[0], transforms[0],
        sitk.sitkNearestNeighbor, 0.0, sitk_moving_imgs[0].GetPixelID()
    )
resampled2 = sitk.Resample(
        sitk_moving_imgs[1], sitk_fixed_imgs[1], transforms[1],
        sitk.sitkNearestNeighbor, 0.0, sitk_moving_imgs[1].GetPixelID()
    )
resampled3 = sitk.Resample(
        sitk_moving_imgs[2], sitk_fixed_imgs[2], transforms[2],
        sitk.sitkNearestNeighbor, 0.0, sitk_moving_imgs[2].GetPixelID()
    )
fixed1 = sitk.GetArrayFromImage(sitk_fixed_imgs[0])
fixed2 = sitk.GetArrayFromImage(sitk_fixed_imgs[1])
fixed3 = sitk.GetArrayFromImage(sitk_fixed_imgs[2])
moved1 = sitk.GetArrayFromImage(resampled1)
moved2 = sitk.GetArrayFromImage(resampled2)
moved3 = sitk.GetArrayFromImage(resampled3)
fig, axes = plt.subplots(nrows=1,ncols=3)
axes[0].imshow(fixed1, cmap="gray")
axes[0].imshow(moved1,cmap='Blues',alpha=0.5)
axes[1].imshow(fixed2, cmap="gray")
axes[1].imshow(moved2,cmap='Blues',alpha=0.5)
axes[2].imshow(fixed3, cmap="gray")
axes[2].imshow(moved3,cmap='Blues',alpha=0.5)

plt.tight_layout()
#plt.savefig("registration_verification_fullmap.png", dpi=150)
plt.show()